# 02 - K-means on a subsample: K sweep  (Stage 2, optional)

Fits `MiniBatchKMeans` for every K in `C.K_LIST` on `C.N_SUB` `complete`
particles, with the radius features standardized (StandardScaler) and weighted
by `C.RADIUS_WEIGHTS`. Elbow / silhouette, a dendrogram of the centroids, and a
numbered centroid map help choose K for notebook 1.5.

Notebook 1.5 does its own fit; the models written here
(`data/kmeans_models/kmeans_k*.pkl`, `scaler.pkl`) are only for this
comparison and are not used by 03 / 04.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # project root: config.py, pipeline.py
import numpy as np
import pandas as pd
import config as C
import pipeline as P
print("project root:", C.PROJECT_ROOT)
print(f"crossing: {C.LAT_CROSS} N | radii (km): {C.RADII_KM}")

In [ ]:
import pickle
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

features = pd.read_parquet(C.FEATURES_FILE)
complete = features[features.status == "complete"]
sub = complete.sample(n=min(C.N_SUB, len(complete)), random_state=C.RANDOM_STATE)

X_raw  = P.build_feature_matrix(sub)
scaler = StandardScaler().fit(X_raw)
W      = P.feature_weight_vector()
X_sub  = scaler.transform(X_raw) * W
with open(C.MODELS_DIR / "scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print(f"complete: {len(complete):,} -> subsample {X_sub.shape}")
print("columns          :", P.feature_columns())
print("std per column   :", np.round(scaler.scale_, 2), "(degrees, before scaling)")
print("radius weights   :", C.RADIUS_WEIGHTS)

## 2.1  Fit every K

In [ ]:
import time
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score

results = []
for k in C.K_LIST:
    t0 = time.time()
    km = MiniBatchKMeans(n_clusters=k, random_state=C.RANDOM_STATE,
                         batch_size=10000, n_init=10, max_iter=300)
    labels = km.fit_predict(X_sub)
    sil = silhouette_score(X_sub, labels, sample_size=C.SILHOUETTE_SAMPLE,
                           random_state=C.RANDOM_STATE)
    sizes = np.bincount(labels, minlength=k)
    with open(C.MODELS_DIR / f"kmeans_k{k}.pkl", "wb") as f:
        pickle.dump(km, f)
    results.append(dict(k=k, inertia=km.inertia_, silhouette=sil,
                        min_size=sizes.min(), max_size=sizes.max()))
    print(f"k={k:4d}  inertia={km.inertia_:.1f}  silhouette={sil:.3f}  "
          f"sizes[{sizes.min()}..{sizes.max()}]  ({time.time()-t0:.1f}s)")
res = pd.DataFrame(results)
res

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(res.k, res.inertia, "o-"); a1.set_xlabel("k"); a1.set_ylabel("inertia"); a1.set_title("Elbow")
a2.plot(res.k, res.silhouette, "o-"); a2.set_xlabel("k"); a2.set_ylabel("silhouette")
a2.set_title("Silhouette vs k")
fig.tight_layout(); plt.show()

## 2.2  Dendrogram of the centroids and a numbered centroid map

Ward clustering of the `INSPECT_K` centroids in the weighted standardized space
(`P.suggest_groups`) gives a first suggestion for merging; the map shows each
centroid from its mean crossing point through the radius exits, numbered at the
outermost radius and haloed in its suggested group colour.

In [ ]:
from scipy.cluster.hierarchy import dendrogram

INSPECT_K = C.K_LIST[0]
N_GROUPS  = 8

with open(C.MODELS_DIR / f"kmeans_k{INSPECT_K}.pkl", "rb") as f:
    km_i = pickle.load(f)
cent_deg = P.centroids_to_degrees(km_i.cluster_centers_ / W, scaler)
lab_i = km_i.predict(X_sub)
start_lon = pd.Series(sub.cross_lon.to_numpy()).groupby(lab_i).mean().reindex(range(INSPECT_K)).to_numpy()
group_map, Z = P.suggest_groups(km_i.cluster_centers_, N_GROUPS)

fig, ax = plt.subplots(figsize=(12, 4))
dendrogram(Z, labels=[str(c) for c in range(INSPECT_K)], ax=ax,
           color_threshold=Z[-(N_GROUPS - 1), 2])
ax.set_title(f"Centroid dendrogram (k={INSPECT_K}); cut -> {N_GROUPS} groups")
ax.set_ylabel("Ward distance (weighted std space)")
plt.show()
print("suggested GROUP_MAP =", group_map)

In [ ]:
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
import cartopy.crs as ccrs, cartopy.feature as cfeature

pc = ccrs.PlateCarree()
cl_cmap  = plt.get_cmap("tab20", INSPECT_K)
grp_ids  = sorted(set(group_map.values()))
grp_cmap = plt.get_cmap("tab10", max(len(grp_ids), 3))
grp_col  = {g: grp_cmap(i) for i, g in enumerate(grp_ids)}

fig = plt.figure(figsize=(11, 8))
ax = plt.axes(projection=pc)
la_all, lo_all = cent_deg[:, 0::2], cent_deg[:, 1::2]
ax.set_extent([min(lo_all.min(), np.nanmin(start_lon)) - 3, lo_all.max() + 3,
               C.LAT_CROSS - 3, la_all.max() + 3], crs=pc)
ax.add_feature(cfeature.LAND, facecolor="0.85"); ax.coastlines(lw=.5)
gl = ax.gridlines(draw_labels=True, lw=.3, color="0.7", alpha=.5)
gl.top_labels = gl.right_labels = False
ax.axhline(C.LAT_CROSS, color="k", lw=.6, ls="--")
for c in range(INSPECT_K):
    lats = np.r_[C.LAT_CROSS, cent_deg[c, 0::2]]
    lons = np.r_[start_lon[c], cent_deg[c, 1::2]]
    ax.plot(lons, lats, ".-", color=cl_cmap(c), lw=1.4, alpha=.8, transform=pc)
    ax.text(lons[-1], lats[-1], str(c), fontsize=10, ha="center", va="center",
            color="white", zorder=4, transform=pc,
            path_effects=[pe.withStroke(linewidth=3, foreground=grp_col[group_map[c]])])
ax.legend(handles=[Line2D([0], [0], marker="o", ls="", mfc=grp_col[g], mec="k",
                          label=f"group {g}") for g in grp_ids], fontsize=8)
ax.set_title(f"Centroids: crossing -> {' -> '.join(f'{r} km' for r in C.RADII_KM)} "
             f"(k={INSPECT_K}); halo = suggested group")
plt.show()